# Voice Agent Architecture

**Module:** 16 — Speech AI

Cascaded STT→LLM→TTS vs speech-to-speech — tools, state, and barge-in.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Diagram cascaded vs speech-to-speech architectures
- List essential components: VAD, endpointing, barge-in, memory
- Design tool calling over voice with confirmations
- Choose when cascade beats S2S and vice versa


## Cascaded Architecture

### Definition
Cascaded voice agents chain ASR → (NLU/LLM/tools) → TTS, with orchestration glue for turns.

### Why it matters
Most production bots still cascade because tools, policies, and transcripts are explicit.

### How it works
Stream partial ASR → decide endpoint → LLM (tools) → stream TTS; cancel on barge-in; persist session state.

### Intuition
Telephone game with specialists — each stage is replaceable.

### Pitfalls
- Waiting for final ASR before any LLM work
- No cancel token for TTS on interrupt

### When to use
Enterprise contact center and tool-heavy agents.


### Cascaded vs speech-to-speech

| | Cascaded | Speech-to-speech |
|--|----------|------------------|
| Intermediate text | Yes | Optional/hidden |
| Tooling | Natural | Needs bridging |
| Latency | Sum of stages | Potentially lower |
| Controllability | High | Emerging |
| Audit transcript | Built-in | Must capture |

```mermaid
flowchart TB
  subgraph casc [Cascaded]
    A[ASR] --> L[LLM + tools] --> T[TTS]
  end
  subgraph s2s [Speech-to-speech]
    X[Audio in] --> M[Realtime model] --> Y[Audio out]
    M --> Tool[Tool bridge]
  end
```


In [ ]:
# Demo 1: cascaded turn state machine
from enum import Enum, auto

class Phase(Enum):
    LISTENING = auto(); THINKING = auto(); SPEAKING = auto(); TOOL = auto()

class VoiceTurn:
    def __init__(self):
        self.phase = Phase.LISTENING
        self.partial = ""
        self.transcript = ""
        self.tts_buffer = ""
    def on_partial(self, text):
        if self.phase == Phase.SPEAKING:
            self.phase = Phase.LISTENING  # barge-in
            self.tts_buffer = ""
        self.partial = text
    def on_final(self, text):
        self.transcript = text
        self.phase = Phase.THINKING
    def on_llm(self, reply, tool=None):
        if tool:
            self.phase = Phase.TOOL
        else:
            self.phase = Phase.SPEAKING
            self.tts_buffer = reply

v = VoiceTurn()
v.on_partial("cancel my")
v.on_final("cancel my order")
v.on_llm("Sure — what's the order id?")
print(v.phase, v.tts_buffer)
v.on_partial("wait")
print("barge-in", v.phase, v.tts_buffer)


In [ ]:
# Demo 2: tool calling with voice confirmation
TOOLS = {
    "cancel_order": lambda oid: {"ok": True, "id": oid},
}

def handle_intent(intent, slots, confirmed=False):
    if intent == "cancel_order":
        if not confirmed:
            return {"speak": f"Confirm cancel order {slots.get('id')}?", "need_confirm": True}
        result = TOOLS["cancel_order"](slots["id"])
        return {"speak": f"Canceled order {result['id']}.", "need_confirm": False}
    return {"speak": "Sorry, I cannot do that.", "need_confirm": False}

print(handle_intent("cancel_order", {"id": "4455"}, confirmed=False))
print(handle_intent("cancel_order", {"id": "4455"}, confirmed=True))


## Essential Components

### Definition
Voice agents need VAD/endpointing, barge-in, turn state, short-term memory, tool policy, and escalation.

### Why it matters
Missing barge-in alone makes bots feel rude and slow.

### How it works
Build a control plane: session store, interrupt channel, timers, allow-listed tools, confidence gates.

### Intuition
A good host: listens, doesn't talk over you, remembers why you called.

### Pitfalls
- LLM-only 'memory' without session store
- Tools without confirmations on money moves

### When to use
All duplex assistants.


In [ ]:
# Demo 3: session memory for voice
from dataclasses import dataclass, field

@dataclass
class Session:
    user_id: str
    slots: dict = field(default_factory=dict)
    history: list = field(default_factory=list)
    def remember(self, k, v): self.slots[k] = v
    def add(self, role, text): self.history.append({"role": role, "text": text})

s = Session("u1")
s.add("user", "Cancel my order")
s.remember("intent", "cancel_order")
s.remember("order_id", "4455")
s.add("assistant", "Confirm cancel 4455?")
print(s)


In [ ]:
# Demo 4: barge-in controller
class BargeInController:
    def __init__(self):
        self.speaking = False
        self.cancel_tts = False
    def start_tts(self):
        self.speaking = True; self.cancel_tts = False
    def on_user_speech(self, vad_prob):
        if self.speaking and vad_prob > 0.7:
            self.cancel_tts = True
            self.speaking = False
            return "interrupt"
        return "continue"

c = BargeInController(); c.start_tts()
print(c.on_user_speech(0.2), c.on_user_speech(0.9), c.cancel_tts)


## Speech-to-Speech

### Definition
S2S models map audio in to audio out (optionally with text mirrors), reducing cascade latency.

### Why it matters
Great for natural conversation; tooling/compliance still need bridges and transcripts.

### How it works
Use realtime APIs; mirror text for audit; gate tools in a side channel; keep human transfer.

### Intuition
Jazz improv vs sheet music — freer, harder to score.

### Pitfalls
- No transcript for regulated industries
- Uncontrolled tool execution from audio

### When to use
Consumer assistants and low-tool chitchat; hybridize for enterprise tools.


In [ ]:
# Demo 5: architecture chooser
def pick_arch(needs: dict) -> str:
    if needs.get("heavy_tools") and needs.get("audit_transcript"):
        return "cascaded (+ optional S2S for chitchat)"
    if needs.get("lowest_latency_chat") and not needs.get("heavy_tools"):
        return "speech_to_speech"
    if needs.get("telephony_ivr"):
        return "cascaded_stream"
    return "cascaded_stream"
print(pick_arch({"heavy_tools": True, "audit_transcript": True}))
print(pick_arch({"lowest_latency_chat": True}))


In [ ]:
# Demo 6: realtime event sketch (illustrative)
import json
events = [
    {"type": "input_audio_buffer.append", "audio": "YOUR_BASE64_PCM_CHUNK"},
    {"type": "input_audio_buffer.commit"},
    {"type": "response.create", "response": {"modalities": ["text", "audio"], "instructions": "Be brief."}},
    {"type": "response.audio.delta", "delta": "BASE64_OUT_CHUNK"},
]
print(json.dumps(events, indent=2)[:500])
print("OPENAI_API_KEY=YOUR_OPENAI_API_KEY for realtime APIs")


### Tool calling over voice — patterns

| Pattern | When |
|---------|------|
| Confirm-then-execute | Money, deletes, PII changes |
| Shadow write | Draft email, show before send |
| Escalation tool | Human warm transfer |
| Read-only tools | Balance inquiry, FAQs |


### Checklist — Voice agent architecture

- [ ] Barge-in cancels TTS
- [ ] Session slot store
- [ ] Confirmations on risky tools
- [ ] Transcript retained per policy
- [ ] Human transfer tool


### Try it yourself — Architecture

1. Add a timeout that auto-reprompts after silence.
2. Implement confirm vocabulary yes/no/affirmatives.
3. Draw sequence diagram for cancel_order with barge-in.

**Stretch:** Prototype websocket events for partial ASR.


### Try it yourself — S2S hybrid

1. Route chitchat to S2S and tools to cascade.
2. Specify how text mirror is stored for audit.


## Knowledge Check

**Q1.** Why confirm before cancel_order?

<details><summary>Answer</summary>

ASR/LLM errors are common; irreversible actions need explicit confirmation.

</details>

**Q2.** What does barge-in require?

<details><summary>Answer</summary>

User speech detection while TTS plays + cancel of outbound audio.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `cascade` | ASR→LLM→TTS pipeline |
| `S2S` | Speech-to-speech model path |
| `barge-in` | User interrupt while bot speaks |
| `endpointing` | End-of-utterance detection |
| `warm transfer` | Hand off to human with context |
| `slots` | Collected entities in session |


## Key Takeaways

- Cascade for tools/audit; S2S for latency/naturalness
- Barge-in and endpointing define UX quality
- Confirm risky tools over voice
- Keep session state outside the LLM


## Production Incident Patterns — voice agents

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Users talk over bot | No barge-in / bad VAD | Tune endpointing; cancel TTS |
| High WER in field | Noise/codec mismatch | Denoise; match sample rate |
| Creepy voice clone | Weak consent policy | Watermark + allow-list |
| 800ms+ dead air | Cascaded STT→LLM→TTS | Speculative TTS; S2S; stream |
| Compliance scare | Raw audio retention | TTL + transcript-only default |

```
Voice control loop:
  mic -> VAD -> ASR partials -> NLU/LLM -> TTS stream -> speaker
                ^                | tools/HITL
                +-- transcripts/metrics/audit --+
```


In [ ]:
# Cross-cutting: never log raw secrets or full audio bytes
import hashlib, json

def audio_audit(user_id: str, wav_bytes: bytes, meta: dict) -> dict:
    return {
        "user_id": user_id,
        "sha256_16": hashlib.sha256(wav_bytes).hexdigest()[:16],
        "nbytes": len(wav_bytes),
        "meta": {k: v for k, v in meta.items() if k not in {"api_key", "authorization"}},
        "topic": "voice agents",
    }

print(json.dumps(audio_audit("u1", b"RIFF....", {"model": "whisper", "api_key": "YOUR_OPENAI_API_KEY"})))


## Mini Case Study — voice agents

**Scenario:** A support org replaces IVR menus with a voice agent. Pilot NPS soars.
**Month 2:** Accents under-served; callers interrupted mid-sentence; recordings retained 2 years.

**Retro questions**
1. What was the latency budget (ASR+LLM+TTS)?
2. Was barge-in tested with noisy headsets?
3. Retention: audio vs transcript vs redacted entities?
4. Which intents require human transfer?

**Design rule:** conversational voice is a real-time distributed system — optimize the path, not only model quality.


In [ ]:
# Cross-cutting: latency budget checker
from dataclasses import dataclass

@dataclass
class VoiceBudget:
    asr_ms: int = 300
    llm_first_token_ms: int = 400
    tts_first_audio_ms: int = 200
    network_ms: int = 100
    def total(self): return self.asr_ms + self.llm_first_token_ms + self.tts_first_audio_ms + self.network_ms
    def ok(self, sla=900): return self.total() <= sla

b = VoiceBudget()
print("voice agents", "total_ms", b.total(), "ok", b.ok())
print("tight", VoiceBudget(500, 600, 300, 150).ok())


### Try it yourself — voice agents ops

1. Draft an on-call runbook bullet list for voice agents when p95 turn latency > SLA.
2. Sketch metrics: WER proxy, barge-in rate, transfer rate, audio retention age.
